# 06 — Fine-tune the forecasting model

Run once: 75,000 examples total across 1Q, 2Q and 4Q. Keep all five tables and enhanced twelve-quarter inputs. This is a new experiment, not a guarantee of lower error. Existing adapters are never overwritten. Notebook 07 compares the new adapter with explicit historical references.


## 1. Mount the project

Use a fresh GPU Colab runtime. This intentionally mounts Google Drive. All three horizons read the same model.yaml; old horizon profiles are not used.


In [ ]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
    import os
    os.chdir("/content/drive/MyDrive/JobAI")
    os.environ["JOBAI_REPO"] = "/content/drive/MyDrive/JobAI"
    os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
    os.environ["HF_HOME"] = "/content/drive/MyDrive/JobAI/.cache/huggingface"
    os.environ["HF_DATASETS_CACHE"] = "/content/drive/MyDrive/JobAI/.cache/huggingface/datasets"
    os.environ["HF_HUB_CACHE"] = "/content/drive/MyDrive/JobAI/.cache/huggingface/hub"
    os.makedirs(os.environ["HF_DATASETS_CACHE"], exist_ok=True)
    os.makedirs(os.environ["HF_HUB_CACHE"], exist_ok=True)
except ImportError:
    pass


## 2. Install libraries and optional Qwen3.5 fast kernels

Keep Colab's managed PyTorch/CUDA/NumPy/pandas unchanged. Kernel installation may compile an extension once. If compilation fails, stop here rather than starting the 75k run on a slow fallback. Restart if Colab requests it, then rerun the mount cell and continue at Section 3. GPU execution is still required to confirm compatibility.


In [ ]:
%pip install --quiet -r requirements-train-colab.txt

import importlib.util, importlib.metadata, subprocess, tempfile
from pathlib import Path
INSTALL_FAST_KERNELS = True
missing = any(importlib.util.find_spec(name) is None for name in ("causal_conv1d", "fla"))
if INSTALL_FAST_KERNELS and missing:
    # Constrain managed packages so a kernel install cannot replace Colab's stack.
    managed = []
    for name in ("torch", "torchvision", "torchaudio", "numpy", "pandas"):
        try:
            managed.append(f"{name}=={importlib.metadata.version(name)}")
        except importlib.metadata.PackageNotFoundError:
            pass
    with tempfile.TemporaryDirectory(prefix="jobai-kernel-install-") as temporary:
        constraints = Path(temporary) / "constraints.txt"
        constraints.write_text("\n".join(managed) + "\n")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-c", str(constraints),
                               "ninja", "packaging", "einops"])
        subprocess.check_call([sys.executable, "-m", "pip", "install", "--no-build-isolation",
                               "-c", str(constraints), "causal-conv1d", "flash-linear-attention"])
print("Kernel availability and ABI are checked next, before model loading.")


## 3. Verify the data, configuration and GPU

Check 03/04/05 agree on the five-table catalog, history length, chronological splits and output checksums. Stale local/Drive outputs cause a clear stop; matching outputs can be reused. Notebook 06 reads no test records.


In [ ]:
import hashlib, importlib.metadata, inspect, json, math, platform, time, uuid
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import yaml
from jobai.forecasting import (PROMPT_VERSION, balanced_sample, digest_json, encode_record,
    prompt_messages, read_json, sha256, verify_pipeline, write_json, input_view, parse_prediction)
from jobai.model_runtime import (attach_language_lora, base_directory, check_fast_kernels,
    load_quantized_base, load_tokenizer, tokenizer_identity)

REPO = Path(os.environ["JOBAI_REPO"]).resolve()
MODEL_CONFIG_PATH = REPO / "configs/model.yaml"
MODEL_CFG = yaml.safe_load(MODEL_CONFIG_PATH.read_text())
EVAL_CFG = yaml.safe_load((REPO / "configs/eval.yaml").read_text())
TRAIN_CFG = MODEL_CFG["training"]
MODEL_ID = MODEL_CFG["candidate_to_run"]
MODEL_CHOICE = next(c for c in MODEL_CFG["candidates"] if c["id"] == MODEL_ID)
HORIZONS = sorted(TRAIN_CFG["training_horizons"])
assert HORIZONS == [1, 2, 4], "This notebook trains one shared adapter; do not load an old horizon profile."
assert TRAIN_CFG["prompt_schema"] == PROMPT_VERSION
panel_card, baseline_manifest = verify_pipeline(REPO, EVAL_CFG)
if not baseline_manifest.get("normalized_inputs"):
    print("Note: older matching baseline manifest has no normalized-input hashes; regenerate 04/05 for full source provenance.")
assert torch.cuda.is_available(), "Select a GPU Colab runtime."
GPU = torch.cuda.get_device_properties(0)
GPU_VRAM_GB = GPU.total_memory / 1024**3
assert GPU_VRAM_GB >= MODEL_CHOICE["minimum_vram_gb"], "GPU is below this experiment's minimum memory policy."
BF16 = torch.cuda.is_bf16_supported()
SEED = int(TRAIN_CFG["seed"])
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
kernel_status = check_fast_kernels(MODEL_ID, MODEL_CHOICE.get("require_fast_kernels", False))
print("Verified shared training:", MODEL_ID, HORIZONS, "budget:", TRAIN_CFG["max_train_examples"])
print("GPU:", GPU.name, round(GPU_VRAM_GB, 1), "GiB; bf16:", BF16)
print("Fast kernel imports:", kernel_status)


## 4. Reuse the base model and tokenizer

Download only when the persistent base cache is missing. A tokenizer fingerprint isolates caches from other models and chat templates. Weight loading into GPU memory happens later, after token-length checks.


In [ ]:
BASE_MODEL_DIR = base_directory(REPO, MODEL_ID, download=True)
print("Using saved base model:", BASE_MODEL_DIR)
tokenizer, processor = load_tokenizer(BASE_MODEL_DIR, MODEL_ID)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"
TOKENIZER_SHA = tokenizer_identity(tokenizer)
BASE_SNAPSHOT = read_json(BASE_MODEL_DIR / "jobai_base_snapshot.json") if (BASE_MODEL_DIR / "jobai_base_snapshot.json").exists() else {}
print("Tokenizer:", type(tokenizer).__name__, "fingerprint:", TOKENIZER_SHA[:16])


## 5. Select and cache 75,000 combined examples

Allocate equal horizon quotas, then balance table/scope/size/volatility within each quota. Keep the small ATP families where the budget permits. No sampling with replacement. Validation is balanced too, but need not retain every ATP row.

The tokenizer creates explicit completion-only labels: prompt tokens are masked, JSON answer and EOS tokens are learned. Every selected record is checked; overlong records stop the run instead of silently losing their answers. Batch size and accumulation are intentionally excluded from this cache key.


In [ ]:
from datasets import Dataset, load_from_disk

cache_spec = {
    "schema": PROMPT_VERSION, "sampling": "balanced_shared_v1", "seed": SEED,
    "model_id": MODEL_ID, "base_config_sha256": sha256(BASE_MODEL_DIR / "config.json"),
    "base_revision": BASE_SNAPSHOT.get("revision"), "tokenizer_sha256": TOKENIZER_SHA,
    "helper_sha256": sha256(REPO / "jobai/forecasting.py"),
    "train_sha256": panel_card["files"]["train"]["sha256"],
    "validation_sha256": panel_card["files"]["validation"]["sha256"],
    "history_quarters": EVAL_CFG["feature_window_quarters"], "horizons": HORIZONS,
    "max_train_examples": TRAIN_CFG["max_train_examples"],
    "max_validation_examples": TRAIN_CFG["max_validation_examples"],
    "priority_tables": TRAIN_CFG["always_include_tables"],
    "max_seq_length": TRAIN_CFG["max_seq_length"],
    "tokenizers_version": importlib.metadata.version("tokenizers"),
    "transformers_version": importlib.metadata.version("transformers"),
}
CACHE_KEY = digest_json(cache_spec)[:24]
CACHE_DIR = REPO / "data/cache/finetune" / CACHE_KEY
CACHE_READY = CACHE_DIR / "complete.json"
if CACHE_READY.is_file():
    cached = read_json(CACHE_READY)
    assert cached["spec"] == cache_spec
    build = CACHE_DIR / cached["build"]
    for name in ("train_sample.parquet", "validation_sample.parquet"):
        assert sha256(build / name) == cached["sample_hashes"][name], "Cached sample changed."
    train_frame = pd.read_parquet(build / "train_sample.parquet")
    val_frame = pd.read_parquet(build / "validation_sample.parquet")
    train_dataset = load_from_disk(str(build / "train_tokens"))
    val_dataset = load_from_disk(str(build / "validation_tokens"))
    print("Reused checked tokenized datasets:", CACHE_DIR)
else:
    full_train = pd.read_json(REPO / panel_card["files"]["train"]["path"], lines=True)
    full_val = pd.read_json(REPO / panel_card["files"]["validation"]["path"], lines=True)
    assert full_train["split"].eq("train").all() and full_val["split"].eq("validation").all()
    train_frame = balanced_sample(full_train, TRAIN_CFG["max_train_examples"], SEED,
                                  HORIZONS, TRAIN_CFG["always_include_tables"])
    val_frame = balanced_sample(full_val, TRAIN_CFG["max_validation_examples"], SEED + 1,
                                HORIZONS, priority_tables=())
    del full_train, full_val
    # A failed build is left isolated; completed caches are never deleted or overwritten.
    build = CACHE_DIR / ("build_" + uuid.uuid4().hex[:12])
    build.mkdir(parents=True, exist_ok=False)
    train_frame.to_parquet(build / "train_sample.parquet", index=False)
    val_frame.to_parquet(build / "validation_sample.parquet", index=False)
    def tokenize(row):
        completion = json.dumps({"target_scaled_change": round(float(row["target_scaled_change"]), 6)})
        history, checked_scale, _, _ = input_view(row, EVAL_CFG["feature_window_quarters"])
        assert np.isclose(checked_scale, row["scale"]) and np.isclose(history[-1], row["last_value"])
        assert np.isclose((row["target_value"] - history[-1]) / checked_scale, row["target_scaled_change"])
        return encode_record(tokenizer, prompt_messages(row, PROMPT_VERSION, EVAL_CFG["feature_window_quarters"]),
                             completion, int(TRAIN_CFG["max_seq_length"]))
    def tokenize_frame(frame):
        source = Dataset.from_pandas(frame, preserve_index=False)
        return source.map(tokenize, remove_columns=source.column_names, load_from_cache_file=False,
                          desc="Tokenize and check every record (no truncation)")
    train_dataset = tokenize_frame(train_frame)
    val_dataset = tokenize_frame(val_frame)
    train_dataset.save_to_disk(str(build / "train_tokens"))
    val_dataset.save_to_disk(str(build / "validation_tokens"))
    write_json(CACHE_READY, {"spec": cache_spec, "build": build.name,
        "sample_hashes": {name: sha256(build / name) for name in ("train_sample.parquet", "validation_sample.parquet")}})
assert set(train_frame.example_id).isdisjoint(set(val_frame.example_id))
assert len(train_dataset) == len(train_frame)
assert TRAIN_CFG["max_train_examples"] is None or len(train_frame) == TRAIN_CFG["max_train_examples"]
assert len(val_dataset) == len(val_frame)
assert TRAIN_CFG["max_validation_examples"] is None or len(val_frame) == TRAIN_CFG["max_validation_examples"]
assert set(train_frame.horizon_q) == set(val_frame.horizon_q) == set(HORIZONS)
lengths = [len(ids) for ids in train_dataset["input_ids"]]
assert max(lengths) <= TRAIN_CFG["max_seq_length"]
for split_dataset in (train_dataset, val_dataset):
    assert all(any(label != -100 for label in labels) for labels in split_dataset["labels"])
print("Training/validation records:", len(train_dataset), "/", len(val_dataset))
print("Training token lengths median/max:", int(np.median(lengths)), "/", max(lengths))
display(train_frame.groupby(["horizon_q", "table_id"]).size().rename("examples").reset_index())


## 6. Load four-bit weights and language-only LoRA

Use the multimodal Qwen3.5 loader for its official checkpoint, but supply only text. Vision weights receive no trainable adapters. Stop if the model is offloaded to CPU rather than silently training slowly.


In [ ]:
model = load_quantized_base(BASE_MODEL_DIR, MODEL_CHOICE["architecture"], BF16)
model, LORA_TARGETS = attach_language_lora(model, MODEL_CFG["lora"], TRAIN_CFG["gradient_checkpointing"])
model.print_trainable_parameters()
print("Language adapter modules:", len(LORA_TARGETS))
print("GPU allocated GiB:", round(torch.cuda.memory_allocated() / 1024**3, 2))


## 7. Create the trainer and a unique run folder

This is completion-only supervised fine-tuning using Transformers Trainer and the explicit labels created above. It avoids SFTTrainer re-tokenization and accidental multimodal data preprocessing. Validation computes loss only, not full-vocabulary predictions.

Recovery checkpoints are separate from the final adapter. To resume an interrupted run, set resume_from_checkpoint to its checkpoint path in model.yaml; the cached dataset and other settings must match.


In [ ]:
from transformers import Trainer, TrainingArguments, DataCollatorForSeq2Seq, TrainerCallback

resume = TRAIN_CFG.get("resume_from_checkpoint")
resume_identity = {"cache_key": CACHE_KEY, "model": MODEL_ID, "lora": MODEL_CFG["lora"],
                   "training": {k: v for k, v in TRAIN_CFG.items() if k != "resume_from_checkpoint"}}
if resume:
    RESUME_PATH = Path(resume)
    if not RESUME_PATH.is_absolute():
        RESUME_PATH = REPO / RESUME_PATH
    assert (RESUME_PATH / "trainer_state.json").is_file(), "Checkpoint does not exist."
    OUTPUT_DIR = RESUME_PATH.parent
    assert read_json(OUTPUT_DIR / "run_setup.json")["resume_identity"] == resume_identity, "Resume configuration/cache differs."
    RUN_ID = OUTPUT_DIR.name
else:
    RESUME_PATH = None
    slug = MODEL_ID.lower().replace("/", "-").replace(".", "-")
    RUN_ID = f"{slug}__shared_enhanced__h1-2-4__{time.strftime('%Y%m%dT%H%M%SZ', time.gmtime())}__{uuid.uuid4().hex[:6]}"
    OUTPUT_DIR = REPO / TRAIN_CFG["output_root"] / RUN_ID
    OUTPUT_DIR.mkdir(parents=True, exist_ok=False)
    write_json(OUTPUT_DIR / "run_setup.json", {"resume_identity": resume_identity, "model_config": MODEL_CFG}, immutable=True)
FINAL_ADAPTER = OUTPUT_DIR / "final_adapter"
assert not FINAL_ADAPTER.exists(), "Completed run already exists; do not overwrite it."
RUN_REPORTS = REPO / "reports/finetune_runs" / RUN_ID
RUN_REPORTS.mkdir(parents=True, exist_ok=True)
effective_batch = TRAIN_CFG["per_device_train_batch_size"] * TRAIN_CFG["gradient_accumulation_steps"]
steps = math.ceil(math.ceil(len(train_dataset) / TRAIN_CFG["per_device_train_batch_size"]) /
                  TRAIN_CFG["gradient_accumulation_steps"] * TRAIN_CFG["num_train_epochs"])
kwargs = dict(output_dir=str(OUTPUT_DIR), num_train_epochs=TRAIN_CFG["num_train_epochs"],
    per_device_train_batch_size=TRAIN_CFG["per_device_train_batch_size"],
    per_device_eval_batch_size=TRAIN_CFG["per_device_eval_batch_size"],
    gradient_accumulation_steps=TRAIN_CFG["gradient_accumulation_steps"],
    learning_rate=float(TRAIN_CFG["learning_rate"]), lr_scheduler_type=TRAIN_CFG["lr_scheduler_type"],
    warmup_steps=math.ceil(steps * TRAIN_CFG["warmup_ratio"]), weight_decay=TRAIN_CFG["weight_decay"],
    logging_steps=TRAIN_CFG["logging_steps"], save_strategy="steps", save_steps=TRAIN_CFG["save_steps"],
    save_total_limit=TRAIN_CFG["save_total_limit"], eval_strategy=TRAIN_CFG["eval_strategy"],
    bf16=bool(BF16), fp16=not bool(BF16), gradient_checkpointing=TRAIN_CFG["gradient_checkpointing"],
    gradient_checkpointing_kwargs={"use_reentrant": False}, optim=TRAIN_CFG["optim"],
    prediction_loss_only=True, remove_unused_columns=False, label_names=["labels"],
    report_to="none", seed=SEED)
if "eval_strategy" not in inspect.signature(TrainingArguments).parameters:
    kwargs["evaluation_strategy"] = kwargs.pop("eval_strategy")
unsupported = set(kwargs) - set(inspect.signature(TrainingArguments).parameters)
assert not unsupported, f"Incompatible training stack, unsupported options: {sorted(unsupported)}"
training_args = TrainingArguments(**kwargs)

class SpeedReport(TrainerCallback):
    def on_train_begin(self, args, state, control, **kwargs):
        self.started = time.monotonic()
        self.initial_step = state.global_step
    def on_step_end(self, args, state, control, **kwargs):
        elapsed_steps = state.global_step - self.initial_step
        if elapsed_steps in (int(TRAIN_CFG["speed_report_step"]), 100):
            seconds = (time.monotonic() - self.started) / elapsed_steps
            remaining = (state.max_steps - state.global_step) * seconds / 3600
            print(f"Measured {seconds:.2f}s/optimizer step; remaining training ~{remaining:.2f}h "
                  f"(validation/saving extra); peak allocated {torch.cuda.max_memory_allocated()/1024**3:.1f} GiB.")

trainer = Trainer(model=model, args=training_args, train_dataset=train_dataset, eval_dataset=val_dataset,
    processing_class=tokenizer,
    data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer, padding=True, label_pad_token_id=-100),
    callbacks=[SpeedReport()])
assert trainer.args.per_device_train_batch_size == TRAIN_CFG["per_device_train_batch_size"]
assert trainer.args.gradient_accumulation_steps == TRAIN_CFG["gradient_accumulation_steps"]
print("Micro-batch:", trainer.args.per_device_train_batch_size, "accumulation:", trainer.args.gradient_accumulation_steps)
print("Effective batch:", effective_batch, "optimizer steps:", steps, "warmup:", kwargs["warmup_steps"])
print("New adapter destination:", FINAL_ADAPTER)


## 8. Train once and save the adapter

This is the only training call. Save the adapter and recovery metadata immediately, before numerical validation. A completed final_adapter folder cannot be overwritten by rerunning this cell.


In [ ]:
assert not FINAL_ADAPTER.exists(), "Final adapter already saved. Continue with validation; do not train again."
started_at = time.time()
train_result = trainer.train(resume_from_checkpoint=str(RESUME_PATH) if RESUME_PATH else None)
training_seconds = time.time() - started_at
trainer.save_model(str(FINAL_ADAPTER))
tokenizer.save_pretrained(FINAL_ADAPTER)
if processor is not None:
    processor.save_pretrained(FINAL_ADAPTER)
pd.DataFrame(trainer.state.log_history).to_csv(RUN_REPORTS / "finetune_training_log.csv", index=False)
run_manifest = {
    "run_id": RUN_ID, "status": "shared_adapter_trained",
    "profile_name": MODEL_CFG["profile_name"], "base_model": MODEL_ID,
    "architecture": MODEL_CHOICE["architecture"], "base_model_commit": BASE_SNAPSHOT.get("revision"),
    "adapter_path": str(FINAL_ADAPTER.relative_to(REPO)), "model_config": MODEL_CFG,
    "prompt_schema": PROMPT_VERSION, "feature_set": TRAIN_CFG["feature_set"],
    "feature_window_quarters": EVAL_CFG["feature_window_quarters"], "training_horizons": HORIZONS,
    "train_examples_used": len(train_dataset), "train_examples": len(train_dataset),
    "validation_examples_used": len(val_dataset), "test_data_used": False,
    "train_split_sha256": panel_card["files"]["train"]["sha256"],
    "validation_split_sha256": panel_card["files"]["validation"]["sha256"],
    "panel_dataset_card_sha256": sha256(REPO / "data/manifests/panel_dataset_card.json"),
    "cache_key": CACHE_KEY, "tokenizer_sha256": TOKENIZER_SHA,
    "train_sample_sha256": sha256(build / "train_sample.parquet"),
    "validation_sample_sha256": sha256(build / "validation_sample.parquet"),
    "train_examples_by_horizon": {str(k): int(v) for k, v in train_frame.groupby("horizon_q").size().items()},
    "train_examples_by_table": train_frame.groupby("table_id").size().astype(int).to_dict(),
    "training_seconds": training_seconds, "gpu": {"name": GPU.name, "vram_gb": GPU_VRAM_GB},
    "kernels": kernel_status, "lora_modules": LORA_TARGETS,
    "packages": {n: importlib.metadata.version(n) for n in
                 ("torch", "transformers", "peft", "datasets", "accelerate", "bitsandbytes")},
}
for package in ("causal-conv1d", "flash-linear-attention"):
    try:
        run_manifest["packages"][package] = importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        pass
write_json(OUTPUT_DIR / "trained_run_metadata.json", run_manifest, immutable=True)
print("Adapter saved:", FINAL_ADAPTER)
print("Training hours:", round(training_seconds / 3600, 2))


## 9. Small numerical validation check

Use up to 96 validation examples per horizon, never test data. Report parsing, MAE and per-series-macro sMAPE. This fixed one-epoch experiment does not select a checkpoint using the test set.


In [ ]:
validation_sample = pd.concat([
    group.sample(min(len(group), int(TRAIN_CFG["validation_forecasts_per_horizon"])), random_state=SEED + int(h))
    for h, group in val_frame.groupby("horizon_q")
], ignore_index=True)
trainer.model.eval()
tokenizer.padding_side = "left"
validation_rows = []
records = validation_sample.to_dict("records")
batch_size = int(TRAIN_CFG["generation_batch_size"])
for start in range(0, len(records), batch_size):
    batch = records[start:start + batch_size]
    prompts = [tokenizer.apply_chat_template(prompt_messages(row, PROMPT_VERSION, EVAL_CFG["feature_window_quarters"]), tokenize=False,
               add_generation_prompt=True, enable_thinking=False) for row in batch]
    inputs = tokenizer(prompts, return_tensors="pt", padding=True, add_special_tokens=False,
                       truncation=False).to(trainer.model.device)
    assert inputs["input_ids"].shape[1] <= TRAIN_CFG["max_seq_length"]
    with torch.inference_mode():
        outputs = trainer.model.generate(**inputs, max_new_tokens=TRAIN_CFG["max_new_tokens"],
            do_sample=False, use_cache=True, pad_token_id=tokenizer.pad_token_id)
    responses = tokenizer.batch_decode(outputs[:, inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    for row, response in zip(batch, responses):
        change = parse_prediction(response)
        prediction = max(0.0, row["last_value"] + change * row["scale"]) if change is not None else np.nan
        target = float(row["target_value"])
        denom = abs(target) + abs(prediction)
        validation_rows.append({"example_id": row["example_id"], "series_id": row["series_id"],
            "horizon_q": row["horizon_q"], "parsed": change is not None, "y_true": target, "y_pred": prediction,
            "MAE": abs(target - prediction), "sMAPE_pct": 0.0 if denom == 0 else 200 * abs(target - prediction) / denom,
            "response": response})
    print(f"Validation: {min(start + batch_size, len(records))}/{len(records)}", end="\r")
validation_predictions = pd.DataFrame(validation_rows)
validation_predictions.to_csv(RUN_REPORTS / "finetune_checkpoint_validation.csv", index=False)
validation_macro = validation_predictions.groupby(["horizon_q", "series_id"])[["MAE", "sMAPE_pct"]].mean().groupby("horizon_q").mean()
display(validation_macro)
print("Parse success:", validation_predictions.parsed.mean())


## 10. Publish the shared-run manifest

Preserve the immutable run record and update one shared latest pointer for Notebook 07. Do not train another horizon profile. Evaluation may still reject the model even when training finishes successfully.


In [ ]:
run_manifest = read_json(OUTPUT_DIR / "trained_run_metadata.json")
run_manifest.update(status="shared_adapter_trained_validation_complete",
    validation_parse_success=float(validation_predictions.parsed.mean()),
    validation_sMAPE_macro=float(validation_macro.sMAPE_pct.mean()) if validation_macro.sMAPE_pct.notna().any() else None,
    validation_MAE_macro=float(validation_macro.MAE.mean()) if validation_macro.MAE.notna().any() else None,
    outputs={name: {"path": str((RUN_REPORTS / name).relative_to(REPO)), "sha256": sha256(RUN_REPORTS / name)}
             for name in ("finetune_training_log.csv", "finetune_checkpoint_validation.csv")})
manifest_path = REPO / "data/manifests/finetune_runs" / f"{RUN_ID}.json"
if manifest_path.exists():
    assert read_json(manifest_path) == run_manifest, "Immutable manifest differs; do not overwrite."
else:
    write_json(manifest_path, run_manifest, immutable=True)
write_json(REPO / "data/manifests/finetune_run_manifest.json", run_manifest)
print("Immutable manifest:", manifest_path)
print("One shared H1/H2/H4 adapter complete. Run Notebook 07; no other training run is required.")
